# Cartes d’identité mutualisées — CSV et SQL

Ce notebook analyse tous les fichiers `*.csv` et `*.sql.txt` du dossier `datas`, puis génère une carte d’identité Markdown par fichier dans `artifacts/ingestion/identities/<AAAAMMJJ>/`.

## Démarche retenue

1. Définir un modèle de résultat commun aux deux formats.
2. Mutualiser les fonctions transverses : charset, taille, affichage des valeurs, rendu Markdown et compte-rendu.
3. Conserver un analyseur spécialisé par format : pandas pour les CSV, SQLAlchemy pour les dumps SQL.
4. Dispatcher automatiquement chaque fichier vers le bon analyseur.
5. Arrêter le lot dès la première erreur et afficher un diagnostic explicite.

> Le dump fourni utilise quelques constructions PostgreSQL. Il est chargé dans une base SQLite en mémoire après deux adaptations ciblées (`TIMESTAMPTZ` et `NOW()`), sans modifier le fichier source.

## 1. Imports et modèle commun

`ColumnProfile` représente une colonne quel que soit son format d’origine. `IdentityAnalysis` porte les métadonnées et profils produits par un analyseur. `IdentityResult` décrit l’artefact finalement écrit. Cette séparation évite de dupliquer le rendu Markdown et le compte-rendu.

In [1]:
from __future__ import annotations

import csv
import io
import re
import statistics
from contextlib import redirect_stdout
from dataclasses import dataclass
from datetime import date, datetime, time
from decimal import Decimal
from pathlib import Path
from typing import Any
from unittest.mock import patch

import pandas as pd
from pandas.api.types import is_bool_dtype, is_numeric_dtype
from sqlalchemy import MetaData, String, cast, create_engine, func, inspect, select
from sqlalchemy.sql.sqltypes import Boolean, Date, DateTime, Float, Integer, Numeric


@dataclass(frozen=True)
class ColumnProfile:
    name: str
    inferred_type: str
    table: str | None = None
    minimum: Any = None
    maximum: Any = None
    mean: Any = None
    median: Any = None


@dataclass(frozen=True)
class TableProfile:
    name: str
    rows: int
    columns: int


@dataclass(frozen=True)
class IdentityAnalysis:
    source: Path
    charset: str
    started_at: datetime
    ended_at: datetime
    rows: int
    columns: int
    column_profiles: tuple[ColumnProfile, ...]
    table_profiles: tuple[TableProfile, ...] = ()


@dataclass(frozen=True)
class IdentityResult:
    source: Path
    artifact: Path
    format: str
    rows: int
    columns: int
    tables: int


## 2. Fonctions mutualisées

Le charset est déterminé avant toute lecture. La détection essaie les encodages du plus spécifique au plus permissif. Les fonctions de présentation garantissent ensuite un Markdown stable pour les deux analyseurs.

In [2]:
def detect_charset(path: Path) -> tuple[str, str]:
    content = path.read_bytes()
    if content.startswith(b"\xef\xbb\xbf"):
        return "UTF-8 BOM", "utf-8-sig"
    for label, codec in (
        ("UTF-8", "utf-8"),
        ("Windows-1252", "cp1252"),
        ("ISO-8859-1", "iso-8859-1"),
    ):
        try:
            content.decode(codec)
            return label, codec
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Impossible de déterminer le charset de {path}")


def human_size(byte_count: int) -> str:
    units = ("o", "Ko", "Mo", "Go", "To")
    size = float(byte_count)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024
    raise AssertionError("Unité de taille inaccessible")


def markdown_escape(value: Any) -> str:
    return str(value).replace("|", r"\|").replace("\n", " ")


def display_value(value: Any) -> str:
    if value is None or (not isinstance(value, (list, tuple, dict)) and pd.isna(value)):
        return "N/A"
    if isinstance(value, datetime):
        return value.isoformat(sep=" ")
    if isinstance(value, (date, time)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return format(value, "f")
    if isinstance(value, float):
        return f"{value:.10g}"
    return str(value)


def artifact_basename(source: Path) -> str:
    return source.name[:-len(".sql.txt")] if source.name.endswith(".sql.txt") else source.stem

## 3. Stratégie CSV avec pandas

Le séparateur est détecté sur un échantillon. L’inférence combine le nom de colonne, le type pandas et la forme des valeurs : les identifiants sont distingués des nombres, tandis que les dates et heures ne sont acceptées que si au moins 90 % des valeurs non nulles respectent le format attendu.

In [3]:
def detect_delimiter(path: Path, encoding: str) -> str:
    with path.open("r", encoding=encoding, newline="") as stream:
        sample = stream.read(16_384)
    if not sample:
        raise ValueError("Le fichier CSV est vide.")
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error as exc:
        raise ValueError("Séparateur CSV indéterminable.") from exc


def infer_csv_column(name: str, series: pd.Series) -> tuple[str, pd.Series | None]:
    normalized_name = name.casefold().strip()
    values = series.dropna().astype("string").str.strip()
    if is_bool_dtype(series.dtype):
        return "BOOLEAN", None

    date_hint = bool(re.search(
        r"(^|_)(date|datetime|timestamp|created_at|updated_at)($|_)", normalized_name
    ))
    date_shape = bool(
        not values.empty
        and values.str.match(r"^\d{4}-\d{2}-\d{2}(?:[ T].*)?$").mean() >= 0.90
    )
    if date_hint or date_shape:
        parsed = pd.to_datetime(values, errors="coerce", format="mixed")
        if not values.empty and parsed.notna().mean() >= 0.90:
            logical_type = (
                "DATETIME"
                if values.str.contains(r"[ T]\d{2}:\d{2}", regex=True).any()
                else "DATE"
            )
            return logical_type, parsed

    time_hint = normalized_name == "time" or normalized_name.endswith("_time")
    time_shape = bool(
        not values.empty
        and values.str.match(r"^\d{2}:\d{2}(?::\d{2}(?:\.\d+)?)?$").mean() >= 0.90
    )
    if time_hint and time_shape:
        return "TIME", values

    identity_hint = (
        normalized_name == "id"
        or normalized_name.endswith("_id")
        or any(word in normalized_name for word in ("badge", "identifier", "identifiant"))
    )
    if identity_hint:
        return "IDENTITY", None
    if is_numeric_dtype(series.dtype):
        return "NUMERIC", None
    return "TEXT", None


def analyze_csv(source: Path) -> IdentityAnalysis:
    started_at = datetime.now().astimezone()
    charset_label, encoding = detect_charset(source)
    delimiter = detect_delimiter(source, encoding)
    frame = pd.read_csv(source, encoding=encoding, sep=delimiter)
    profiles: list[ColumnProfile] = []

    for raw_name in frame.columns:
        name = str(raw_name)
        series = frame[raw_name]
        logical_type, parsed = infer_csv_column(name, series)
        values = None
        if logical_type == "NUMERIC":
            values = pd.to_numeric(series, errors="coerce").dropna()
        elif logical_type in {"DATE", "DATETIME", "TIME"} and parsed is not None:
            values = parsed.dropna()

        if values is not None and not values.empty:
            profiles.append(ColumnProfile(
                name=name,
                inferred_type=logical_type,
                minimum=values.min(),
                maximum=values.max(),
                mean=values.mean() if logical_type == "NUMERIC" else None,
                median=values.median() if logical_type == "NUMERIC" else None,
            ))
        else:
            profiles.append(ColumnProfile(name=name, inferred_type=logical_type))

    return IdentityAnalysis(
        source=source.resolve(),
        charset=charset_label,
        started_at=started_at,
        ended_at=datetime.now().astimezone(),
        rows=len(frame),
        columns=len(frame.columns),
        column_profiles=tuple(profiles),
    )

## 4. Stratégie SQL avec SQLAlchemy

Le dump est exécuté dans une base SQLite en mémoire, puis SQLAlchemy réfléchit le schéma réel. Une clé primaire, une clé étrangère ou une colonne terminant par `_id` est classée `IDENTITY`. Les statistiques sont ensuite calculées sur les données chargées. Les nombres de lignes et colonnes du fichier correspondent aux sommes de toutes ses tables ; le détail par table reste présent dans la carte.

In [4]:
def sqlite_compatible_sql(sql: str) -> str:
    sql = re.sub(r"\bTIMESTAMPTZ\b", "TIMESTAMP", sql, flags=re.IGNORECASE)
    return re.sub(r"\bNOW\s*\(\s*\)", "CURRENT_TIMESTAMP", sql, flags=re.IGNORECASE)


def infer_sql_column(column: Any) -> str:
    name = column.name.casefold()
    if column.primary_key or column.foreign_keys or name == "id" or name.endswith("_id"):
        return "IDENTITY"
    if isinstance(column.type, Boolean):
        return "BOOLEAN"
    if isinstance(column.type, DateTime):
        return "DATETIME"
    if isinstance(column.type, Date):
        return "DATE"
    if isinstance(column.type, (Integer, Numeric, Float)):
        return "NUMERIC"
    return "TEXT"


def profile_sql_column(connection: Any, table: Any, column: Any) -> ColumnProfile:
    logical_type = infer_sql_column(column)
    common = {"table": table.name, "name": column.name, "inferred_type": logical_type}
    if logical_type == "NUMERIC":
        values = list(connection.scalars(select(column).where(column.is_not(None))))
        if values:
            return ColumnProfile(
                **common,
                minimum=min(values),
                maximum=max(values),
                mean=statistics.fmean(values),
                median=statistics.median(values),
            )
    elif logical_type in {"DATE", "DATETIME"}:
        minimum, maximum = connection.execute(
            select(func.min(cast(column, String)), func.max(cast(column, String)))
        ).one()
        if minimum is not None:
            return ColumnProfile(**common, minimum=minimum, maximum=maximum)
    return ColumnProfile(**common)


def analyze_sql(source: Path) -> IdentityAnalysis:
    started_at = datetime.now().astimezone()
    charset_label, encoding = detect_charset(source)
    sql = source.read_text(encoding=encoding)
    if not sql.strip():
        raise ValueError("Le fichier SQL est vide.")

    engine = create_engine("sqlite+pysqlite:///:memory:")
    try:
        raw_connection = engine.raw_connection()
        try:
            raw_connection.executescript(sqlite_compatible_sql(sql))
            raw_connection.commit()
        finally:
            raw_connection.close()

        metadata = MetaData()
        metadata.reflect(bind=engine)
        if not metadata.tables:
            raise ValueError("Aucune table trouvée dans le fichier SQL.")

        table_profiles: list[TableProfile] = []
        column_profiles: list[ColumnProfile] = []
        with engine.connect() as connection:
            for table_name in inspect(engine).get_table_names():
                table = metadata.tables[table_name]
                row_count = connection.scalar(select(func.count()).select_from(table)) or 0
                table_profiles.append(TableProfile(table_name, row_count, len(table.columns)))
                column_profiles.extend(
                    profile_sql_column(connection, table, column) for column in table.columns
                )
    finally:
        engine.dispose()

    return IdentityAnalysis(
        source=source.resolve(),
        charset=charset_label,
        started_at=started_at,
        ended_at=datetime.now().astimezone(),
        rows=sum(item.rows for item in table_profiles),
        columns=sum(item.columns for item in table_profiles),
        column_profiles=tuple(column_profiles),
        table_profiles=tuple(table_profiles),
    )

## 5. Rendu et orchestration communs

Le rendu ne connaît pas le lecteur utilisé : il reçoit uniquement un `IdentityAnalysis`. Le traitement est volontairement séquentiel afin de garantir l’arrêt immédiat à la première erreur et un compte-rendu non ambigu.

In [5]:
def render_identity(analysis: IdentityAnalysis, output_root: Path) -> Path:
    day_directory = output_root / analysis.started_at.strftime("%Y%m%d")
    day_directory.mkdir(parents=True, exist_ok=True)
    artifact = day_directory / (
        f"{artifact_basename(analysis.source)}_{analysis.started_at.strftime('%H%M%S')}.md"
    )
    lines = [
        f"* Source: {analysis.source}",
        f"* Filsize: {human_size(analysis.source.stat().st_size)}",
        f"* ProcessBeginAt: {analysis.started_at.strftime('%H:%M:%S')}",
        f"* ProcessEndAt: {analysis.ended_at.strftime('%H:%M:%S')}",
        f"* Charset: {analysis.charset}",
        f"* Rows Number: {analysis.rows}",
        f"* Cols Number: {analysis.columns}",
    ]

    if analysis.table_profiles:
        lines.extend([
            f"* Tables Number: {len(analysis.table_profiles)}",
            "", "## Tables", "",
            "| Table | Rows Number | Cols Number |",
            "| --- | ---: | ---: |",
        ])
        lines.extend(
            f"| {markdown_escape(item.name)} | {item.rows} | {item.columns} |"
            for item in analysis.table_profiles
        )

    has_tables = bool(analysis.table_profiles)
    lines.extend(["", "## Column list", ""])
    if has_tables:
        lines.extend(["| Table | Colname | Infered Type |", "| --- | --- | --- |"])
        lines.extend(
            f"| {markdown_escape(item.table)} | {markdown_escape(item.name)} | {item.inferred_type} |"
            for item in analysis.column_profiles
        )
    else:
        lines.extend(["| Colname | Infered Type |", "| --- | --- |"])
        lines.extend(
            f"| {markdown_escape(item.name)} | {item.inferred_type} |"
            for item in analysis.column_profiles
        )

    lines.extend(["", "## Numeric and temporal statistics", ""])
    prefix = "| Table " if has_tables else "|"
    if has_tables:
        lines.extend([
            "| Table | Col name | Infered Type | Min value | Max value | Mean value | Median value |",
            "| --- | --- | --- | ---: | ---: | ---: | ---: |",
        ])
    else:
        lines.extend([
            "| Col name | Infered Type | Min value | Max value | Mean value | Median value |",
            "| --- | --- | ---: | ---: | ---: | ---: |",
        ])

    statistical = [
        item for item in analysis.column_profiles
        if item.inferred_type in {"NUMERIC", "DATE", "DATETIME", "TIME"}
    ]
    for item in statistical:
        values = ([item.table] if has_tables else []) + [
            item.name, item.inferred_type, item.minimum, item.maximum, item.mean, item.median
        ]
        lines.append("| " + " | ".join(
            markdown_escape(display_value(value)) for value in values
        ) + " |")
    if not statistical:
        empty_cells = 7 if has_tables else 6
        lines.append("| _Aucune colonne numérique ou temporelle_ " + "| N/A " * (empty_cells - 1) + "|")

    artifact.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return artifact.resolve()


def process_file(source: Path, output_root: Path) -> IdentityResult:
    if source.name.endswith(".sql.txt"):
        file_format = "SQL"
        analysis = analyze_sql(source)
    elif source.suffix.casefold() == ".csv":
        file_format = "CSV"
        analysis = analyze_csv(source)
    else:
        raise ValueError(f"Format non pris en charge : {source.name}")
    artifact = render_identity(analysis, output_root)
    return IdentityResult(
        source=analysis.source, artifact=artifact, format=file_format,
        rows=analysis.rows, columns=analysis.columns, tables=len(analysis.table_profiles),
    )


def generate_identities(
    input_directory: Path = Path("datas"),
    output_root: Path = Path("artifacts/ingestion/identities"),
) -> list[IdentityResult]:
    sources = sorted(
        path for path in input_directory.iterdir()
        if path.is_file() and (path.suffix.casefold() == ".csv" or path.name.endswith(".sql.txt"))
    )
    if not sources:
        raise FileNotFoundError(f"Aucun fichier CSV ou SQL trouvé dans {input_directory.resolve()}")

    results: list[IdentityResult] = []
    for source in sources:
        try:
            results.append(process_file(source, output_root))
        except Exception as exc:
            print(f"ERREUR pendant le traitement de {source.resolve()}")
            print(f"{type(exc).__name__}: {exc}")
            raise RuntimeError(f"Traitement interrompu sur {source.name}") from exc

    print(f"Traitement terminé avec succès : {len(results)} fichier(s) traité(s).")
    for result in results:
        table_text = f", {result.tables} table(s)" if result.tables else ""
        print(
            f"- {result.source.name} [{result.format}] : {result.rows} ligne(s), "
            f"{result.columns} colonne(s){table_text} -> {result.artifact}"
        )
    return results

## 6. Exécution du traitement

Une seule fonction traite désormais les deux formats et retourne des objets homogènes. Le tableau final facilite la vérification du lot.

In [6]:
results = generate_identities(
    input_directory=Path("datas"),
    output_root=Path("artifacts/ingestion/identities"),
)

summary = pd.DataFrame([
    {
        "Fichier": item.source.name,
        "Format": item.format,
        "Tables": item.tables or "N/A",
        "Lignes": item.rows,
        "Colonnes": item.columns,
        "Artefact": str(item.artifact),
    }
    for item in results
])
summary

Traitement terminé avec succès : 3 fichier(s) traité(s).
- machine.sql.txt [SQL] : 130 ligne(s), 22 colonne(s), 2 table(s) -> D:\indusense\artifacts\ingestion\identities\20261120\machine_171002.md
- releves_incidents.csv [CSV] : 1245 ligne(s), 18 colonne(s) -> D:\indusense\artifacts\ingestion\identities\20261120\releves_incidents_171002.md
- telemetry.csv [CSV] : 135626 ligne(s), 7 colonne(s) -> D:\indusense\artifacts\ingestion\identities\20261120\telemetry_171002.md


,Fichier,Format,Tables,Lignes,Colonnes,Artefact
0,machine.sql.txt,SQL,2,130,22,D:\indusense\artifacts\ingestion\identities\20...
1,releves_incidents.csv,CSV,N/A,1245,18,D:\indusense\artifacts\ingestion\identities\20...
2,telemetry.csv,CSV,N/A,135626,7,D:\indusense\artifacts\ingestion\identities\20...


## 7. Contrôles automatiques

Les assertions ci-dessous verrouillent les dimensions connues des sources, l’existence des artefacts et le comportement d’arrêt sur erreur. Pour ce dernier contrôle, une panne d’analyseur est simulée sans créer de fichier parasite.

In [7]:
expected_dimensions = {
    "releves_incidents.csv": (1245, 18),
    "telemetry.csv": (135626, 7),
    "machine.sql.txt": (130, 22),
}
assert len(results) == 3
for item in results:
    assert (item.rows, item.columns) == expected_dimensions[item.source.name]
    assert item.artifact.is_file()

captured = io.StringIO()
with patch(__name__ + ".process_file", side_effect=ValueError("erreur simulée")):
    try:
        with redirect_stdout(captured):
            generate_identities(Path("datas"), Path("artifacts/ingestion/identities"))
    except RuntimeError as exc:
        assert "Traitement interrompu sur machine.sql.txt" in str(exc)
        assert "ValueError: erreur simulée" in captured.getvalue()
    else:
        raise AssertionError("Le traitement aurait dû s’arrêter sur l’erreur simulée.")

print("Contrôles réussis : dimensions, artefacts et arrêt sur erreur validés.")

Contrôles réussis : dimensions, artefacts et arrêt sur erreur validés.


## Synthèse des choix

- **Mutualisation au niveau du modèle et du rendu** : les différences de lecture ne contaminent pas la génération des cartes.
- **Stratégies spécialisées** : pandas reste adapté aux fichiers tabulaires ; SQLAlchemy apporte la réflexion du schéma SQL.
- **Base SQL en mémoire** : aucune base persistante ni fichier intermédiaire n’est créé.
- **Traitement séquentiel** : le contrat d’arrêt immédiat sur erreur est respecté.
- **Statistiques ciblées** : moyenne et médiane uniquement pour les nombres ; minimum et maximum pour les nombres et valeurs temporelles.
- **Notebook autonome** : tout le code utile est visible, expliqué et exécutable sans dépendre du module applicatif existant.